# Cross-Device Validation â€” IBM Marrakesh

## Goal

Reproduce the Î”H and isotropy results from IBM Fez on a **second independent backend**
(IBM Marrakesh) to confirm that the entropy reduction signal is a structural property
of the inference framework, not a single-device artifact.

## Three codes (same as Fez experiment)

| Code | Qubits | Stabilizers | Hypotheses | Circuits |
|------|--------|-------------|------------|----------|
| HaPPY [[5,1,3]] | 5+1 | 4 | 16 | 32 |
| Steane [[7,1,3]] | 7+1 | 6 | 22 | 44 |
| Shor [[9,1,3]] | 9+1 | 8 | 22 | 44 |

**Total: 120 circuits Ã— 8192 shots â‰ˆ 2 min QPU time**

## Resume capability

Set `RESUME_JOB_IDS` dict below to skip submission and retrieve previous runs.

In [10]:
"""Cell 1: Imports & Configuration"""

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "qiskit", "qiskit-aer", "qiskit-ibm-runtime",
                       "matplotlib", "numpy", "-q"])

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from itertools import combinations
import time, os

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Pauli, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# ============================================================
# CONFIGURATION
# ============================================================
SHOTS = 8192
SEED = 42
BACKEND_NAME = "ibm_marrakesh"

IBM_TOKEN = "5GI3W65AZds2mLbWCaXyjr59CjZSFTBdfR8XJJsEywS8"
INSTANCE = "open-instance"
CHANNEL = "ibm_quantum_platform"

# Resume from previous jobs (set to None to submit fresh)
# Keys: "happy", "steane", "shor"
RESUME_JOB_IDS = {
    "happy":  None,
    "steane": None,
    "shor":   None,
}

np.random.seed(SEED)
print(f"Target: {BACKEND_NAME}, {SHOTS} shots/circuit")
print("Imports OK")

Target: ibm_marrakesh, 8192 shots/circuit
Imports OK


In [5]:
"""Cell 2: Code Definitions (all three codes)"""

def anticommutes(a, b):
    return not (a == "I" or b == "I" or a == b)

def syndrome_of(error_str, stabilizers):
    syn = []
    for stab in stabilizers:
        n = sum(anticommutes(e, s) for e, s in zip(error_str, stab))
        syn.append(n % 2)
    return tuple(syn)

def compute_logical_states(stabilizers, n_data, z_l, x_l):
    """Compute |0_L> and |1_L> via stabilizer projection."""
    dim = 2 ** n_data
    proj = np.eye(dim, dtype=np.complex128)
    for stab in stabilizers:
        P = Pauli(stab[::-1]).to_matrix()
        proj = proj @ ((np.eye(dim) + P) / 2)

    psi = None
    for basis_idx in range(dim):
        v = proj @ np.eye(dim)[basis_idx]
        norm = np.linalg.norm(v)
        if norm > 1e-6:
            psi = v / norm
            break
    assert psi is not None, "Failed to find codespace vector"

    Z_L_mat = Pauli(z_l[::-1]).to_matrix()
    z_vals = Z_L_mat @ psi
    v0 = (psi + z_vals) / 2
    v1 = (psi - z_vals) / 2
    n0, n1 = np.linalg.norm(v0), np.linalg.norm(v1)

    if n0 > 1e-12:
        state0 = v0 / n0
    else:
        state0 = v1 / n1
        v1 = v0
        n1 = n0

    if n1 > 1e-12:
        state1 = v1 / n1
    else:
        X_L_mat = Pauli(x_l[::-1]).to_matrix()
        state1 = X_L_mat @ state0

    # Verify
    z0 = state0.conj() @ Z_L_mat @ state0
    z1 = state1.conj() @ Z_L_mat @ state1
    assert abs(z0.real - 1.0) < 0.01, f"Z_L|0_L> != +1: {z0}"
    assert abs(z1.real + 1.0) < 0.01, f"Z_L|1_L> != -1: {z1}"

    return state0, state1

# â”€â”€ HaPPY [[5,1,3]] â”€â”€
HAPPY_STABS = ["XZZXI", "IXZZX", "XIXZZ", "ZXIXZ"]
HAPPY_N_DATA, HAPPY_N_ANC = 5, 4
HAPPY_Z_L = "ZZZZZ"
HAPPY_X_L = "XXXXX"

errors_happy, labels_happy = ["IIIII"], ["I"]
for q in range(5):
    for p in "XYZ":
        e = list("IIIII"); e[q] = p
        errors_happy.append("".join(e))
        labels_happy.append(f"{p}{q}")
HAPPY_N_HYP = len(errors_happy)  # 16

happy_s0, happy_s1 = compute_logical_states(
    HAPPY_STABS, HAPPY_N_DATA, HAPPY_Z_L, HAPPY_X_L)

# â”€â”€ Steane [[7,1,3]] â”€â”€
STEANE_STABS = ["IIIXXXX", "IXXIIXX", "XIXIXIX",
                "IIIZZZZ", "IZZIIZZ", "ZIZIZIZ"]
STEANE_N_DATA, STEANE_N_ANC = 7, 6
STEANE_Z_L = "ZZZZZZZ"
STEANE_X_L = "XXXXXXX"

errors_steane, labels_steane = ["IIIIIII"], ["I"]
for q in range(7):
    for p in "XYZ":
        e = list("IIIIIII"); e[q] = p
        errors_steane.append("".join(e))
        labels_steane.append(f"{p}{q}")
STEANE_N_HYP = len(errors_steane)  # 22

steane_s0, steane_s1 = compute_logical_states(
    STEANE_STABS, STEANE_N_DATA, STEANE_Z_L, STEANE_X_L)

# â”€â”€ Shor [[9,1,3]] â”€â”€
SHOR_STABS = [
    "ZZIIIIIII", "IZZIIIIII", "IIIZZIIII",
    "IIIIZZIII", "IIIIIIZZI", "IIIIIIIZZ",
    "XXXXXXIII", "IIIXXXXXX"
]
SHOR_N_DATA, SHOR_N_ANC = 9, 8
SHOR_Z_L = "ZIIZIIIZI"
SHOR_X_L = "XXXXXXXXX"

errors_shor, labels_shor = ["IIIIIIIII"], ["I"]
for q in range(9):
    for p in "XY":
        e = list("IIIIIIIII"); e[q] = p
        errors_shor.append("".join(e))
        labels_shor.append(f"{p}{q}")
for q in [0, 3, 6]:
    e = list("IIIIIIIII"); e[q] = "Z"
    errors_shor.append("".join(e))
    labels_shor.append(f"Z{q}")
SHOR_N_HYP = len(errors_shor)  # 22

shor_s0, shor_s1 = compute_logical_states(
    SHOR_STABS, SHOR_N_DATA, SHOR_Z_L, SHOR_X_L)

# Verify syndrome uniqueness for all codes
for name, errs, stabs in [("HaPPY", errors_happy, HAPPY_STABS),
                           ("Steane", errors_steane, STEANE_STABS),
                           ("Shor", errors_shor, SHOR_STABS)]:
    syns = [syndrome_of(e, stabs) for e in errs]
    assert len(set(syns)) == len(syns), f"{name}: duplicate syndromes!"

print(f"HaPPY:  {HAPPY_N_HYP} hypotheses, {HAPPY_N_ANC} stabilizers")
print(f"Steane: {STEANE_N_HYP} hypotheses, {STEANE_N_ANC} stabilizers")
print(f"Shor:   {SHOR_N_HYP} hypotheses, {SHOR_N_ANC} stabilizers")
print(f"Total circuits: {2*(HAPPY_N_HYP + STEANE_N_HYP + SHOR_N_HYP)}")
print("All logical states and syndromes verified")

HaPPY:  16 hypotheses, 4 stabilizers
Steane: 22 hypotheses, 6 stabilizers
Shor:   22 hypotheses, 8 stabilizers
Total circuits: 120
All logical states and syndromes verified


In [6]:
"""Cell 3: Circuit Builder"""

def perturb_state(vec):
    """Add tiny non-Clifford perturbation to prevent transpiler collapse."""
    v = vec.copy()
    v[0] *= np.exp(1j * 1e-10)
    v /= np.linalg.norm(v)
    return v

def build_circuit(state_vec, error_str, n_data, stabilizers):
    """Build: initialize -> error -> syndrome extraction -> measurement."""
    n_anc = len(stabilizers)
    qr = QuantumRegister(n_data, "data")
    anc = QuantumRegister(1, "anc")
    cr_syn = ClassicalRegister(n_anc, "syn")
    cr_out = ClassicalRegister(n_data, "out")
    qc = QuantumCircuit(qr, anc, cr_syn, cr_out)

    # 1. State preparation
    qc.initialize(state_vec, qr)

    # 2. Error injection
    for q, p in enumerate(error_str):
        if p == "X": qc.x(qr[q])
        elif p == "Y": qc.y(qr[q])
        elif p == "Z": qc.z(qr[q])
    qc.barrier()

    # 3. Syndrome extraction (sequential ancilla reuse)
    for s_idx, stab in enumerate(stabilizers):
        qc.reset(anc[0])
        qc.h(anc[0])
        for q, pauli in enumerate(stab):
            if pauli == "X":
                qc.cx(anc[0], qr[q])
            elif pauli == "Z":
                qc.cz(anc[0], qr[q])
            elif pauli == "Y":
                qc.sdg(qr[q])
                qc.cx(anc[0], qr[q])
                qc.s(qr[q])
        qc.h(anc[0])
        qc.measure(anc[0], cr_syn[s_idx])
    qc.barrier()

    # 4. Measure data qubits
    qc.measure(qr, cr_out)
    return qc

def build_code_circuits(s0, s1, errors, labels, n_data, stabilizers):
    """Build all circuits for one code (n_hyp Ã— 2 logical states)."""
    s0p, s1p = perturb_state(s0), perturb_state(s1)
    circuits, circuit_labels = [], []
    for ls in [0, 1]:
        sv = s0p if ls == 0 else s1p
        for err_str, err_label in zip(errors, labels):
            qc = build_circuit(sv, err_str, n_data, stabilizers)
            circuits.append(qc)
            circuit_labels.append(f"L{ls}_{err_label}")
    return circuits, circuit_labels

# Build all circuits
happy_circs, happy_labels = build_code_circuits(
    happy_s0, happy_s1, errors_happy, labels_happy,
    HAPPY_N_DATA, HAPPY_STABS)

steane_circs, steane_labels = build_code_circuits(
    steane_s0, steane_s1, errors_steane, labels_steane,
    STEANE_N_DATA, STEANE_STABS)

shor_circs, shor_labels = build_code_circuits(
    shor_s0, shor_s1, errors_shor, labels_shor,
    SHOR_N_DATA, SHOR_STABS)

print(f"HaPPY:  {len(happy_circs)} circuits ({happy_circs[0].num_qubits} qubits)")
print(f"Steane: {len(steane_circs)} circuits ({steane_circs[0].num_qubits} qubits)")
print(f"Shor:   {len(shor_circs)} circuits ({shor_circs[0].num_qubits} qubits)")
print(f"Total:  {len(happy_circs) + len(steane_circs) + len(shor_circs)} circuits")

HaPPY:  32 circuits (6 qubits)
Steane: 44 circuits (8 qubits)
Shor:   44 circuits (10 qubits)
Total:  120 circuits


In [7]:
"""Cell 4: Noiseless Validation on Aer"""

from qiskit_aer import AerSimulator

sim = AerSimulator()

all_codes = [
    ("HaPPY",  happy_circs,  happy_labels,  errors_happy,  labels_happy,  HAPPY_STABS),
    ("Steane", steane_circs, steane_labels, errors_steane, labels_steane, STEANE_STABS),
    ("Shor",   shor_circs,  shor_labels,   errors_shor,   labels_shor,   SHOR_STABS),
]

for code_name, circs, clabels, errs, elabels, stabs in all_codes:
    n_data = len(errs[0])
    t_circs = transpile(circs, sim, optimization_level=1)
    result_sim = sim.run(t_circs, shots=256, seed_simulator=SEED).result()

    all_ok = True
    for i, label in enumerate(clabels):
        counts = result_sim.get_counts(i)
        syn_counts = Counter()
        for bitstring, count in counts.items():
            bits = bitstring.replace(" ", "")
            syn_str = bits[n_data:]
            syn = tuple(int(b) for b in reversed(syn_str))
            syn_counts[syn] += count
        dominant = syn_counts.most_common(1)[0]
        err_label = label.split("_")[1]
        err_idx = elabels.index(err_label)
        expected = syndrome_of(errs[err_idx], stabs)
        if dominant[0] != expected or dominant[1] < 250:
            print(f"FAIL {code_name}: {label} expected {expected}, got {dominant[0]} ({dominant[1]}/256)")
            all_ok = False

    status = "OK" if all_ok else "FAIL"
    print(f"{code_name}: {len(circs)} circuits â€” {status}")

HaPPY: 32 circuits â€” OK
Steane: 44 circuits â€” OK
Shor: 44 circuits â€” OK


In [ ]:
"""Cell 5: Connect to IBM Marrakesh & Transpile"""

service = QiskitRuntimeService(
    channel=CHANNEL, token=IBM_TOKEN, instance=INSTANCE
)
backend = service.backend(BACKEND_NAME)
print(f"Connected to {backend.name}")
print(f"  Qubits: {backend.num_qubits}")

# Transpile each code separately (different qubit counts)
code_hw_circuits = {}

for code_name, circs in [("happy", happy_circs),
                          ("steane", steane_circs),
                          ("shor", shor_circs)]:
    print(f"\nTranspiling {code_name} ({len(circs)} circuits)...")
    t0 = time.time()
    hw = transpile(circs, backend=backend, optimization_level=3,
                   seed_transpiler=SEED)
    elapsed = time.time() - t0

    depths = [qc.depth() for qc in hw]
    n2q = []
    for qc in hw:
        ops = qc.count_ops()
        n2q.append(sum(v for k, v in ops.items()
                       if k in ["cx", "ecr", "cz", "rzx", "rzz",
                                "xx_plus_yy", "xx_minus_yy", "cp"]))

    code_hw_circuits[code_name] = hw
    print(f"  {elapsed:.1f}s â€” depth: {min(depths)}-{max(depths)} "
          f"(mean {np.mean(depths):.0f}), 2Q: {min(n2q)}-{max(n2q)} "
          f"(mean {np.mean(n2q):.0f})")

    if max(n2q) == 0:
        print(f"  WARNING: 0 two-qubit gates â€” do NOT submit!")
    else:
        print(f"  OK â€” entangling gates present")

In [ ]:
"""Cell 6: Submit to IBM Marrakesh (per-code jobs with resume)"""

sampler = Sampler(mode=backend)
results = {}
job_ids_out = {}

for code_name, hw_circs, clabels in [
    ("happy",  code_hw_circuits["happy"],  happy_labels),
    ("steane", code_hw_circuits["steane"], steane_labels),
    ("shor",   code_hw_circuits["shor"],   shor_labels),
]:
    resume_id = RESUME_JOB_IDS.get(code_name)

    if resume_id is not None:
        print(f"{code_name}: resuming from job {resume_id}")
        job = service.job(resume_id)
        result = job.result()
        print(f"  Retrieved {len(result)} PUBs")
    else:
        print(f"{code_name}: submitting {len(hw_circs)} circuits Ã— {SHOTS} shots...")
        t0 = time.time()
        pubs = [(qc, [], SHOTS) for qc in hw_circs]
        job = sampler.run(pubs)
        job_id = job.job_id()
        print(f"  Job submitted: {job_id}")
        print(f"  Waiting for results...")
        result = job.result()
        elapsed = time.time() - t0
        print(f"  Done in {elapsed:.0f}s")
        resume_id = job_id

    results[code_name] = result
    job_ids_out[code_name] = resume_id

print(f"\n=== Job IDs (save these for resume) ===")
print(f'RESUME_JOB_IDS = {{')
for k, v in job_ids_out.items():
    print(f'    "{k}":  "{v}",')
print(f'}}')

In [13]:
"""Cell 7: Extract & Save Raw Data"""

def extract_bitstrings(pub_result, register_name):
    data = getattr(pub_result.data, register_name)
    bitstrings = data.get_bitstrings()
    arr = np.array([[int(b) for b in s[::-1]] for s in bitstrings], dtype=int)
    return arr

# Extract all hw_data dicts
hw_data_all = {}

for code_name, result, clabels in [
    ("happy",  results["happy"],  happy_labels),
    ("steane", results["steane"], steane_labels),
    ("shor",   results["shor"],   shor_labels),
]:
    hw_data = {}
    for i, label in enumerate(clabels):
        pub_result = result[i]
        syn_arr = extract_bitstrings(pub_result, "syn")
        out_arr = extract_bitstrings(pub_result, "out")
        hw_data[label] = {"syndrome": syn_arr, "data": out_arr}

    hw_data_all[code_name] = hw_data

    # Print trivial-syndrome rate for no-error circuits
    for state in ["0", "1"]:
        syn = hw_data[f"L{state}_I"]["syndrome"]
        trivial = np.all(syn == 0, axis=1).mean()
        print(f"{code_name} |{state}_L> no error: {trivial*100:.1f}% trivial syndrome")

# Save .npz files
# Try Google Drive first (Colab), fall back to local data/
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    save_dir = "."
    os.makedirs(save_dir, exist_ok=True)
except ImportError:
    save_dir = os.path.join(os.path.dirname(os.path.abspath(".")), "data")
    if not os.path.isdir(save_dir):
        save_dir = os.path.join(os.getcwd(), "data")
    os.makedirs(save_dir, exist_ok=True)

filenames = {
    "happy":  "happy_553_marrakech.npz",
    "steane": "steane_713_marrakech.npz",
    "shor":   "shor_913_marrakech.npz",
}

for code_name, fname in filenames.items():
    clabels = {"happy": happy_labels, "steane": steane_labels, "shor": shor_labels}[code_name]
    hw_data = hw_data_all[code_name]
    save_dict = {}
    for i, label in enumerate(clabels):
        save_dict[f"pub{i}_syn"] = hw_data[label]["syndrome"]
        save_dict[f"pub{i}_out"] = hw_data[label]["data"]
    save_dict["circuit_labels"] = np.array(clabels)

    path = os.path.join(save_dir, fname)
    np.savez_compressed(path, **save_dict)
    print(f"Saved: {path}")

print(f"\nAll data saved to {save_dir}")

In [14]:
"""Cell 8: Î”H Computation â€” All Three Codes"""

def entropy_bits(p):
    p = np.asarray(p, dtype=np.float64)
    p = p[p > 0]
    return -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

def normalize_logits(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - np.max(logits)
    probs = np.exp(logits)
    Z = np.sum(probs)
    return probs / Z if Z > 1e-300 else np.ones_like(probs) / len(probs)

def compute_delta_H(hw_data_in, labels, n_hyp, n_anc, p_no_err=0.7,
                     stab_subset=None):
    """Compute Î”H with 2-fold cross-validation."""
    if stab_subset is None:
        stab_subset = list(range(n_anc))
    n_sub = len(stab_subset)
    n_syn_space = 2 ** n_sub

    log_prior_depol = np.zeros(n_hyp)
    log_prior_depol[0] = np.log(p_no_err)
    log_prior_depol[1:] = np.log((1 - p_no_err) / (n_hyp - 1))
    log_prior_uniform = np.zeros(n_hyp)
    powers = 2 ** np.arange(n_sub - 1, -1, -1)

    all_dH = []

    for state_label in ["0", "1"]:
        state_data = {}
        for err_idx, err_label in enumerate(labels):
            key = f"L{state_label}_{err_label}"
            if key in hw_data_in:
                state_data[err_idx] = hw_data_in[key]["syndrome"][:, stab_subset]

        if not state_data:
            continue

        n_shots = len(list(state_data.values())[0])
        mid = n_shots // 2

        for test_sl, train_sl in [(slice(0, mid), slice(mid, None)),
                                   (slice(mid, None), slice(0, mid))]:
            log_lik = np.full((n_hyp, n_syn_space), np.log(1.0 / n_syn_space))

            for h_idx in state_data:
                train_syn = state_data[h_idx][train_sl]
                keys = (train_syn * powers).sum(axis=1)
                counts = np.zeros(n_syn_space)
                for k in keys:
                    counts[k] += 1
                dist = (counts + 1) / (counts.sum() + n_syn_space)
                log_lik[h_idx] = np.log(dist)

            for err_idx in state_data:
                test_syn = state_data[err_idx][test_sl]
                syn_keys = (test_syn * powers).sum(axis=1)
                for syn_idx in syn_keys:
                    p_fwd = normalize_logits(log_lik[:, syn_idx] + log_prior_uniform)
                    p_dbci = normalize_logits(log_lik[:, syn_idx] + log_prior_depol)
                    all_dH.append(entropy_bits(p_fwd) - entropy_bits(p_dbci))

    return np.mean(all_dH), np.std(all_dH) / np.sqrt(len(all_dH))

# Compute Î”H for all three codes on Marrakech
code_configs = [
    ("HaPPY",  hw_data_all["happy"],  labels_happy,  HAPPY_N_HYP,  HAPPY_N_ANC),
    ("Steane", hw_data_all["steane"], labels_steane, STEANE_N_HYP, STEANE_N_ANC),
    ("Shor",   hw_data_all["shor"],   labels_shor,   SHOR_N_HYP,  SHOR_N_ANC),
]

dH_marrakech = {}
print(f"{'Code':>10} {'Î”H (bits)':>15} {'SEM':>10}")
print("-" * 40)

for code_name, hw_data, labels, n_hyp, n_anc in code_configs:
    dH, sem = compute_delta_H(hw_data, labels, n_hyp, n_anc)
    dH_marrakech[code_name] = (dH, sem)
    print(f"{code_name:>10} {dH:>15.4f} {sem:>10.4f}")

print(f"\nAll three codes show Î”H > 0 on {BACKEND_NAME}.")

      Code       Î”H (bits)        SEM
----------------------------------------
     HaPPY          1.4911     0.0014
    Steane          2.2134     0.0004
      Shor          1.9733     0.0009

All three codes show Î”H > 0 on ibm_marrakesh.


In [16]:
"""Cell 9: Isotropy Analysis â€” CV per code (vectorized)"""

def _entropy_vec(p_arr):
    """Vectorized entropy: p_arr shape (N, n_hyp) -> shape (N,)."""
    safe = np.where(p_arr > 0, p_arr, 1.0)
    return -np.sum(p_arr * np.log2(safe), axis=1)

def _norm_logits(logits_arr):
    """Vectorized softmax: logits_arr shape (N, n_hyp) -> shape (N, n_hyp)."""
    shifted = logits_arr - logits_arr.max(axis=1, keepdims=True)
    probs = np.exp(shifted)
    Z = probs.sum(axis=1, keepdims=True)
    return probs / np.where(Z > 1e-300, Z, 1.0)

def compute_delta_H_fast(hw_data_in, labels, n_hyp, n_anc, p_no_err=0.7,
                          stab_subset=None):
    """Vectorized Î”H computation."""
    if stab_subset is None:
        stab_subset = list(range(n_anc))
    n_sub = len(stab_subset)
    n_syn_space = 2 ** n_sub
    sub = list(stab_subset)

    log_prior_depol = np.zeros(n_hyp)
    log_prior_depol[0] = np.log(p_no_err)
    log_prior_depol[1:] = np.log((1 - p_no_err) / (n_hyp - 1))
    log_prior_uniform = np.zeros(n_hyp)
    powers = 2 ** np.arange(n_sub - 1, -1, -1)

    all_dH = []

    for state_label in ["0", "1"]:
        state_data = {}
        for err_idx, err_label in enumerate(labels):
            key = f"L{state_label}_{err_label}"
            if key in hw_data_in:
                state_data[err_idx] = hw_data_in[key]["syndrome"][:, sub]

        if not state_data:
            continue

        n_shots = len(list(state_data.values())[0])
        mid = n_shots // 2

        for test_sl, train_sl in [(slice(0, mid), slice(mid, None)),
                                   (slice(mid, None), slice(0, mid))]:
            log_lik = np.full((n_hyp, n_syn_space), np.log(1.0 / n_syn_space))

            for h_idx in state_data:
                train_syn = state_data[h_idx][train_sl]
                keys = (train_syn @ powers).astype(int)
                counts = np.bincount(keys, minlength=n_syn_space).astype(float)
                dist = (counts + 1) / (counts.sum() + n_syn_space)
                log_lik[h_idx] = np.log(dist)

            for err_idx in state_data:
                test_syn = state_data[err_idx][test_sl]
                syn_keys = (test_syn @ powers).astype(int)
                ll_batch = log_lik[:, syn_keys].T  # (n_shots, n_hyp)

                p_fwd = _norm_logits(ll_batch + log_prior_uniform)
                p_dbci = _norm_logits(ll_batch + log_prior_depol)

                dH_batch = _entropy_vec(p_fwd) - _entropy_vec(p_dbci)
                all_dH.append(dH_batch)

    all_dH = np.concatenate(all_dH)
    return float(np.mean(all_dH)), float(np.std(all_dH) / np.sqrt(len(all_dH)))

def compute_isotropy(hw_data, labels, n_hyp, n_anc):
    """Compute CV of Î”H across all C(n_anc, k) subsets for k=1..n_anc."""
    results = {}
    for k in range(1, n_anc + 1):
        subs = list(combinations(range(n_anc), k))
        dH_vals = []
        for sub in subs:
            dH, _ = compute_delta_H_fast(hw_data, labels, n_hyp, n_anc,
                                          stab_subset=list(sub))
            dH_vals.append(dH)
        mean_dH = np.mean(dH_vals)
        std_dH = np.std(dH_vals, ddof=0)
        cv = std_dH / mean_dH if mean_dH > 0 else 0
        results[k] = {"mean": mean_dH, "std": std_dH, "cv": cv,
                       "min": np.min(dH_vals), "max": np.max(dH_vals)}
    return results

isotropy_marrakech = {}

for code_name, hw_data, labels, n_hyp, n_anc in code_configs:
    print(f"\n=== {code_name} isotropy ({BACKEND_NAME}) ===")
    print(f"{'k':>3} {'Subsets':>8} {'Mean Î”H':>10} {'CV':>8} {'Min':>10} {'Max':>10}")
    print("-" * 55)
    t0 = time.time()
    iso = compute_isotropy(hw_data, labels, n_hyp, n_anc)
    isotropy_marrakech[code_name] = iso
    for k in sorted(iso.keys()):
        r = iso[k]
        n_subs = len(list(combinations(range(n_anc), k)))
        print(f"{k:>3} {n_subs:>8} {r['mean']:>10.4f} {r['cv']:>7.1%} "
              f"{r['min']:>10.4f} {r['max']:>10.4f}")
    print(f"  ({time.time()-t0:.1f}s)")


=== HaPPY isotropy (ibm_marrakesh) ===
  k  Subsets    Mean Î”H       CV        Min        Max
-------------------------------------------------------
  1        4     1.8303    9.0%     1.5452     1.9447
  2        6     1.7157   10.9%     1.5199     1.9137
  3        4     1.6030   10.0%     1.4952     1.8794
  4        1     1.4911    0.0%     1.4911     1.4911
  (10.5s)

=== Steane isotropy (ibm_marrakesh) ===
  k  Subsets    Mean Î”H       CV        Min        Max
-------------------------------------------------------
  1        6     2.2577    0.1%     2.2543     2.2598
  2       15     2.2538    0.1%     2.2477     2.2573
  3       20     2.2483    0.1%     2.2399     2.2530
  4       15     2.2406    0.1%     2.2345     2.2445
  5        6     2.2296    0.1%     2.2265     2.2323
  6        1     2.2134    0.0%     2.2134     2.2134
  (43.5s)

=== Shor isotropy (ibm_marrakesh) ===
  k  Subsets    Mean Î”H       CV        Min        Max
----------------------------------------

In [ ]:
"""Cell 10: Cross-Device Comparison â€” Fez vs Marrakesh

Load Fez reference data and compare Î”H side by side.
"""

# Load Fez data from .npz files
def find_npz(filename):
    candidates = [
        f"/content/data/{filename}",
        os.path.join(os.path.dirname(os.path.abspath(".")), "data", filename),
        os.path.join(os.getcwd(), "data", filename),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} not found in any candidate directory")

def load_hw_data(npz_path, circuit_labels):
    raw = np.load(npz_path, allow_pickle=True)
    hw_data = {}
    for i, label in enumerate(circuit_labels):
        hw_data[label] = {
            "syndrome": raw[f"pub{i}_syn"].astype(int),
            "data":     raw[f"pub{i}_out"].astype(int),
        }
    return hw_data

# Build circuit label lists for Fez data
fez_happy_labels = [f"L{s}_{l}" for s in [0, 1] for l in labels_happy]
fez_steane_labels = [f"L{s}_{l}" for s in [0, 1] for l in labels_steane]
fez_shor_labels = [f"L{s}_{l}" for s in [0, 1] for l in labels_shor]

dH_fez = {}
print("Loading Fez reference data...")

for code_name, fname, clabels, elabels, n_hyp, n_anc in [
    ("HaPPY",  "happy_553.npz",  fez_happy_labels,  labels_happy,  HAPPY_N_HYP,  HAPPY_N_ANC),
    ("Steane", "steane_713.npz", fez_steane_labels, labels_steane, STEANE_N_HYP, STEANE_N_ANC),
    ("Shor",   "shor_913.npz",  fez_shor_labels,   labels_shor,   SHOR_N_HYP,  SHOR_N_ANC),
]:
    try:
        path = find_npz(fname)
        hw_data = load_hw_data(path, clabels)
        dH, sem = compute_delta_H(hw_data, elabels, n_hyp, n_anc)
        dH_fez[code_name] = (dH, sem)
        print(f"  {code_name} (Fez): Î”H = {dH:.4f} Â± {sem:.4f}")
    except FileNotFoundError:
        print(f"  {code_name}: Fez data not found â€” skipping")

# Side-by-side comparison
print(f"\n{'='*65}")
print(f"{'CROSS-DEVICE COMPARISON':^65}")
print(f"{'='*65}")
print(f"{'Code':>10} {'IBM Fez':>20} {'IBM Marrakesh':>20} {'Î”':>10}")
print(f"{'':<10} {'Î”H Â± SEM':>20} {'Î”H Â± SEM':>20} {'(bits)':>10}")
print("-" * 65)

for code_name in ["HaPPY", "Steane", "Shor"]:
    fez_str = "â€”"
    mar_str = "â€”"
    diff_str = "â€”"

    if code_name in dH_fez:
        dH_f, sem_f = dH_fez[code_name]
        fez_str = f"{dH_f:.4f} Â± {sem_f:.4f}"

    if code_name in dH_marrakech:
        dH_m, sem_m = dH_marrakech[code_name]
        mar_str = f"{dH_m:.4f} Â± {sem_m:.4f}"

    if code_name in dH_fez and code_name in dH_marrakech:
        diff = dH_m - dH_f
        diff_str = f"{diff:+.4f}"

    print(f"{code_name:>10} {fez_str:>20} {mar_str:>20} {diff_str:>10}")

# Conservative conclusion
print(f"\nConclusion: Î”H > 0 reproduces on all three codes across devices.")
print(f"Steane and Shor are quantitatively stable (Î” < 0.04 bits).")
print(f"HaPPY shows a larger device-dependent shift (+0.26 bits),")
print(f"consistent with its greater sensitivity to the noise profile.")
print(f"The qualitative pattern â€” Î”H > 1 bit on all codes, isotropy")
print(f"gap between holographic and non-holographic â€” is device-independent.")

Loading Fez reference data...
  HaPPY (Fez): ΔH = 1.2299 ± 0.0012
  Steane (Fez): ΔH = 2.2104 ± 0.0004
  Shor (Fez): ΔH = 2.0083 ± 0.0008

                     CROSS-DEVICE COMPARISON                     
      Code           IBM Fez        IBM Marrakesh          Δ
                    ΔH ± SEM             ΔH ± SEM     (bits)
-----------------------------------------------------------------
     HaPPY      1.2299 ± 0.0012      1.4911 ± 0.0014    +0.2613
    Steane      2.2104 ± 0.0004      2.2134 ± 0.0004    +0.0030
      Shor      2.0083 ± 0.0008      1.9733 ± 0.0009    -0.0349

Conclusion: ΔH > 0 reproduces on all three codes across devices.
Steane and Shor are quantitatively stable (Δ < 0.04 bits).
HaPPY shows a larger device-dependent shift (+0.26 bits),
consistent with its greater sensitivity to the noise profile.
The qualitative pattern — ΔH > 1 bit on all codes, isotropy
gap between holographic and non-holographic — is device-independent.


In [ ]:
"""Cell 11: Cross-Device Figure"""

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Î”H bar chart: Fez vs Marrakesh
ax = axes[0]
codes = ["HaPPY", "Steane", "Shor"]
x = np.arange(len(codes))
w = 0.35

fez_vals = [dH_fez.get(c, (0, 0))[0] for c in codes]
fez_sems = [dH_fez.get(c, (0, 0))[1] for c in codes]
mar_vals = [dH_marrakech.get(c, (0, 0))[0] for c in codes]
mar_sems = [dH_marrakech.get(c, (0, 0))[1] for c in codes]

ax.bar(x - w/2, fez_vals, w, yerr=fez_sems, label="IBM Fez",
       color="steelblue", alpha=0.8, capsize=4, edgecolor="black", linewidth=0.5)
ax.bar(x + w/2, mar_vals, w, yerr=mar_sems, label="IBM Marrakesh",
       color="coral", alpha=0.8, capsize=4, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(codes)
ax.set_ylabel("Î”H (bits)", fontsize=12)
ax.set_title("(a) Cross-Device Î”H Comparison", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

# (b) Isotropy CV comparison at matched fractions
ax = axes[1]
# Fez reference CVs (from isotropy_gap_analysis.ipynb)
fez_cv = {
    "HaPPY":  {1: 6.6, 2: 8.2, 3: 7.4},
    "Steane": {1: 0.1, 2: 0.2, 3: 0.2, 4: 0.2, 5: 0.2},
    "Shor":   {1: 1.9, 2: 2.6, 3: 3.1, 4: 3.3, 5: 3.3, 6: 3.0, 7: 2.3},
}

colors = {"HaPPY": "steelblue", "Steane": "darkorange", "Shor": "forestgreen"}

for code_name in codes:
    n_anc = {"HaPPY": HAPPY_N_ANC, "Steane": STEANE_N_ANC, "Shor": SHOR_N_ANC}[code_name]
    if code_name in isotropy_marrakech:
        iso = isotropy_marrakech[code_name]
        fracs = [k / n_anc for k in iso if k < n_anc]
        cvs = [iso[k]["cv"] * 100 for k in iso if k < n_anc]
        ax.plot(fracs, cvs, "o-", color=colors[code_name], lw=2, ms=7,
                label=f"{code_name} (Marrakesh)")

    # Fez reference as dashed
    if code_name in fez_cv:
        fracs_f = [k / n_anc for k in fez_cv[code_name]]
        cvs_f = list(fez_cv[code_name].values())
        ax.plot(fracs_f, cvs_f, "x--", color=colors[code_name], lw=1.5, ms=7,
                alpha=0.5, label=f"{code_name} (Fez)")

ax.set_xlabel("Fraction of stabilizers (k/n)", fontsize=12)
ax.set_ylabel("CV of Î”H (%)", fontsize=12)
ax.set_title("(b) Isotropy: Fez vs Marrakesh", fontsize=13)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig("cross_device_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: cross_device_comparison.png")

In [ ]:
"""Cell 12: Paper-Ready Text (copy into main.tex Â§4.1)

Prints the cross-device paragraph with actual numbers filled in.
"""

print("=" * 70)
print("TEXT FOR main.tex Â§4.1 â€” replace the TODO placeholder")
print("=" * 70)

# Format results with 3 decimal places
lines = []
for code_name in ["HaPPY", "Steane", "Shor"]:
    if code_name in dH_marrakech:
        dH, sem = dH_marrakech[code_name]
        lines.append(f"{code_name}: ${dH:.3f} \\pm {sem:.3f}$~bits")

marrakech_qubits = backend.num_qubits
marrakech_proc = "Heron r2"  # update if different

text = f"""To test whether the observed entropy reduction is specific to a single
device, we repeated the full experimental protocol on IBM Marrakesh
({marrakech_qubits}-qubit {marrakech_proc}). The qualitative pattern reproduces:
all three codes show $\\Delta H > 1$~bit ({'; '.join(lines)}),
and the isotropy gap persists (HaPPY CV 9.0--10.9\\%, Steane CV 0.1\\%,
Shor CV 2.8--4.3\\%). The absolute $\\Delta H$ values vary modestly across
devices, consistent with differing noise profiles; Steane and Shor are
quantitatively stable ($\\Delta < 0.04$~bits), while HaPPY shows a larger
device-dependent shift ($+0.26$~bits), consistent with its greater
sensitivity to the noise environment. This cross-platform consistency
indicates that the entropy reduction signal and the isotropy gap reflect
structural properties of the codes rather than single-device artifacts."""

print(text)
print()

# Also print isotropy comparison
print("Isotropy CV at ~75% fraction:")
for code_name in ["HaPPY", "Steane", "Shor"]:
    n_anc = {"HaPPY": HAPPY_N_ANC, "Steane": STEANE_N_ANC, "Shor": SHOR_N_ANC}[code_name]
    k_match = round(0.75 * n_anc)
    if code_name in isotropy_marrakech and k_match in isotropy_marrakech[code_name]:
        cv = isotropy_marrakech[code_name][k_match]["cv"] * 100
        print(f"  {code_name} k={k_match}/{n_anc}: CV = {cv:.1f}%")

To test whether the observed entropy reduction is specific to a single
device, we repeated the full experimental protocol on IBM Marrakesh
(156-qubit Heron r2). The qualitative pattern reproduces:
all three codes show $\Delta H > 1$~bit (HaPPY: .491 \pm 0.001$~bits; Steane: .213 \pm 0.000$~bits; Shor: .973 \pm 0.001$~bits),
and the isotropy gap persists (HaPPY CV 9.0--10.9\%, Steane CV 0.1\%,
Shor CV 2.8--4.3\%). The absolute $\Delta H$ values vary modestly across
devices, consistent with differing noise profiles; Steane and Shor are
quantitatively stable ($\Delta < 0.04$~bits), while HaPPY shows a larger
device-dependent shift ($+0.26$~bits), consistent with its greater
sensitivity to the noise environment. This cross-platform consistency
indicates that the entropy reduction signal and the isotropy gap reflect
structural properties of the codes rather than single-device artifacts.

Isotropy CV at ~75% fraction:
  HaPPY k=3/4: CV = 10.0%
  Steane k=4/6: CV = 0.1%
  Shor k=6/8: CV = 3.7